In [1]:
import os, numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

base = "/kaggle/input/datasets/yummyooo123/www2025-mmctr-data/MicroLens_1M_MMCTR"
img_dir = f"{base}/item_images/item_images"
ids = pd.read_parquet(f"{base}/item_emb.parquet", columns=["item_id"])["item_id"].sort_values().tolist()

name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(name)
model = CLIPModel.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()

class ImgDS(Dataset):
    def __len__(self): return len(ids)
    def __getitem__(self, i):
        try:
            img = Image.open(f"{img_dir}/{ids[i]}.jpg").convert("RGB")
            return processor(images=img, return_tensors="pt")["pixel_values"][0]
        except Exception:
            return torch.zeros(3, 224, 224)

loader = DataLoader(ImgDS(), batch_size=256, num_workers=4)

feats = []
with torch.no_grad():
    for k, x in enumerate(loader):
        out = model.vision_model(pixel_values=x.cuda().half())
        f = model.visual_projection(out.pooler_output)
        f = torch.nn.functional.normalize(f.float(), dim=-1)
        feats.append(f.cpu().numpy())
        if k % 50 == 0: print(k, "/", len(loader), flush=True)

feats = np.concatenate(feats)
np.save("/kaggle/working/clip_img_512.npy", feats)
np.save("/kaggle/working/item_ids.npy", np.array(ids))
print(feats.shape)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

0 / 359
50 / 359
100 / 359
150 / 359
200 / 359
250 / 359
300 / 359
350 / 359
(91717, 512)


In [2]:
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.decomposition import PCA, TruncatedSVD

base = "/kaggle/input/datasets/yummyooo123/www2025-mmctr-data/MicroLens_1M_MMCTR"

clip = np.load("/kaggle/working/clip_img_512.npy")
ids = np.load("/kaggle/working/item_ids.npy")

info = pd.read_parquet(f"{base}/MicroLens_1M_x1/item_info.parquet")
info = info.set_index("item_id").loc[ids]
tags = np.stack(info["item_tags"].values)
n_tags = tags.max() + 1
rows = np.repeat(np.arange(len(ids)), tags.shape[1])
cols = tags.reshape(-1)
mask = cols > 0
M = sparse.csr_matrix((np.ones(mask.sum()), (rows[mask], cols[mask])),
                      shape=(len(ids), n_tags))
tag_vec = TruncatedSVD(n_components=32, random_state=0).fit_transform(M)
tag_vec /= (np.linalg.norm(tag_vec, axis=1, keepdims=True) + 1e-8)

X = np.concatenate([clip, 0.5 * tag_vec], axis=1)
pca = PCA(n_components=128, random_state=0)
emb = pca.fit_transform(X).astype(np.float32)
emb /= emb.std()
print("variance expliquée :", pca.explained_variance_ratio_.sum())
print(emb.shape)

full = np.zeros((int(ids.max()) + 1, 128), dtype=np.float32)
full[ids] = emb
orig = pd.read_parquet(f"{base}/MicroLens_1M_x1/item_info.parquet")
orig = orig.sort_values("item_id").reset_index(drop=True)
assert len(orig) == len(full)
orig["item_emb_d128"] = list(full)
orig.to_parquet("/kaggle/working/item_info_mine.parquet")
print("fichier sauvegardé", orig.shape)

variance expliquée : 0.834503361451796
(91717, 128)
fichier sauvegardé (91718, 3)
